# Stage 2 — Information Extraction

Loads the **Stage 1 cohort** and runs the information extraction agent on each discharge note.

**LLM backend:** set `LLM_PROVIDER` in `pipeline_config.py` — `"ollama"` or `"api"`

**Input:** `data/.../cohort/cohort.pkl`  
**Output:** `data/.../stage_02_information_extraction/information_extractions.json`

**Next:** Run `stage_03_symptom_tree.ipynb`


## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ollama_agents.py").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from cohort_selection import load_cohort
from llm_client import LLMNotAvailableError, check_llm
from ollama_agents import information_extraction_agent, warn_if_slow_model
from pipeline_config import (
    IE_CHECKPOINT_JSON,
    get_llm_config,
    print_pipeline_banner,
)
from pipeline_io import load_ie_checkpoint, save_ie_checkpoint, save_ie_results

print_pipeline_banner()
LLM_CONFIG = get_llm_config()

ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
warn_if_slow_model(model_info, LLM_CONFIG.provider)
print(f"LLM ready — {LLM_CONFIG.provider}: {model_info}")
print(f"Timeout: {LLM_CONFIG.timeout_seconds}s | note chars: {LLM_CONFIG.max_note_chars}")

cohort_df = load_cohort()
print(f"Loaded cohort: {len(cohort_df)} admissions, {cohort_df['patient_id'].nunique()} patients")


## Run Information Extraction

In [ ]:
checkpoint = load_ie_checkpoint()
rows = list(checkpoint.values())
done_hadm = set(checkpoint.keys())
print(f"Resuming: {len(done_hadm)} already done, {len(cohort_df) - len(done_hadm)} remaining")

for _, row in cohort_df.iterrows():
    hadm_id = int(row["hadm_id"])
    if hadm_id in done_hadm:
        continue

    print(f"IE hadm_id={hadm_id} (subject_id={row['subject_id']})...")
    try:
        extracted = information_extraction_agent(row["clinical_note"], config=LLM_CONFIG)
    except TimeoutError as exc:
        save_ie_checkpoint(rows)
        raise TimeoutError(
            f"{exc}\nCheckpoint saved ({len(rows)} admissions) → {IE_CHECKPOINT_JSON}\n"
            "Re-run this cell to resume."
        ) from exc

    method = extracted.pop("_method", "unknown")
    record = {
        "patient_id": row["patient_id"],
        "admission_id": row["admission_id"],
        "subject_id": int(row["subject_id"]),
        "hadm_id": hadm_id,
        "note_type": row["note_type"],
        "primary_icd_code": row["primary_icd_code"],
        "primary_dx_title": row["primary_dx_title"],
        "ground_truth_icd10": row["ground_truth_icd10"],
        "ground_truth_dx_titles": row["ground_truth_dx_titles"],
        "n_diagnoses": int(row["n_diagnoses"]),
        "extraction_method": method,
        "symptom_count": len(extracted.get("symptoms", [])),
        "diagnosis_count": len(extracted.get("diagnoses_mentioned", [])),
        "extracted": extracted,
    }
    rows.append(record)
    done_hadm.add(hadm_id)
    save_ie_checkpoint(rows)

results_df = pd.DataFrame(rows)
print(f"\nDone — {len(results_df)} admissions")
results_df[["patient_id", "admission_id", "extraction_method", "symptom_count", "diagnosis_count"]]


## Save Results

In [ ]:
out = save_ie_results(results_df)
print(f"Saved → {out}")
